In [0]:
from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
)

import os
import delta_table_creator, transform, user_defined_functions 

# spark.conf.set("spark.databricks.io.cache.enabled", "false")

In [0]:
# Import the module
import sys
sys.path.append("../aws_utils")

import aws_utils
# Get Environment Prefix

env = aws_utils.get_env_prefix()

In [0]:
schema=(
    "DPaymentScheduleId string"
    ",SourceSystem string"
    ",PortfolioName string"
    ",PortfolioKey int"
    ",IngestionTimestamp timestamp"
    ",BatchId string"
    ",DPaymentScheduleKey string"
    ",SettlementKey string"
    ",SettlementCaseRecordKey int"
    ",PayTolerance int"
    ",PayPercent decimal(12,6)"
    ",PayPercentAmount decimal(18,2)"
    ",RecordKey int"
    ",ExposureKey string"
    ",PartyKey string"
    ",AgreementKey string"
    ",InstallmentDate timestamp"
    ",InstallmentTypeCode string"
    ",InstallmentType string"
    ",InstallmentNumber int"
    ",RequestSN int"
    ",PaymentScheduleTypeCode string"
    ",PaymentScheduleType string"
    ",OverdueDays int"
    ",InstallmentStatusCode string"
    ",InstallmentStatus string"
    ",InstallmentAmount Decimal(38,7)"
    ",InterestAmount Decimal(19,2)"
    ",CapitalAmount Decimal(19,2)"
    ",CommissionAmount Decimal(19,2)"
    ",ExpenseAmount Decimal(19,2)"
    ",InstallmentAmountPaid Decimal(19,2)"
    ",CapitalAmountPaid Decimal(19,2)"
    ",ExpenseAmountPaid Decimal(19,2)"
    ",CommissionAmountPaid Decimal(19,2)"
    ",InterestAmountPaid Decimal(19,2)"
    ",RemainingAmount Decimal(19,2)"
    ",DownPaymentFlag boolean"
    ",ImplementationFlag boolean"
    ",BusinessDate date"
)

partition_columns = ["BusinessDate"]
run_date = dbutils.widgets.get("run_date")
unity = dbutils.widgets.get("unity")
table_name = "DPaymentSchedule"
derivative_table_path = ""

# ================== Create Delta Table ==================
if unity == "True":
    derivative_table_path = f"`{env}-cdh`.derivativeactive.{table_name}"
    if not DeltaTable.isDeltaTable(spark, derivative_table_path):
        delta_table_creator.create_unity_delta_table(
            spark, derivative_table_path, partition_columns, schema
        )
else:
    derivative_table_path = f"s3://{env}-ec1-cepal-derivative-zone/{table_name}"
    if not DeltaTable.isDeltaTable(spark, derivative_table_path):
        delta_table_creator.create_delta_table(
            spark, derivative_table_path, partition_columns, schema
        )


#================== Delete From Delta Table ==================

transform.delete_from_unity_table(
    spark,
    derivative_table_path,
    run_date,  # type: ignore
)

In [0]:
portfolio = spark.sql(f""" select * from `{env}-cdh`.primary.portfolio where BusinessDate = {run_date!r}""")
portfolio.createOrReplaceTempView("portfolio")

settlementexposure = spark.sql(f""" select * from `{env}-cdh`.primary.settlementexposure where BusinessDate = {run_date!r}""")
settlementexposure.createOrReplaceTempView("settlementexposure")

In [0]:
# Create temp view for each source table, based on the run date

paymentscheduleprofits = spark.sql(f""" select
    p.ExposureKey, 
    p.InstallmentDate, 
    COALESCE(NULLIF(TRIM(p.InstallmentTypeCode), ''), 'N/A') AS InstallmentTypeCode,
    COALESCE(NULLIF(TRIM(p.InstallmentType), ''), 'N/A') AS InstallmentType,
    p.InstallmentNumber,
    p.RequestSN,
    p.PaymentScheduleTypeCode,
    p.PaymentScheduleType,
    p.OverdueDays, 
    COALESCE(NULLIF(TRIM(p.InstallmentStatusCode), ''), 'N/A') AS InstallmentStatusCode,
    COALESCE(NULLIF(TRIM(p.InstallmentStatus), ''), 'N/A') AS InstallmentStatus,
    ABS(p.InstallmentAmount) AS InstallmentAmount,
    ABS(p.InterestAmount) AS InterestAmount,
    ABS(p.CapitalAmount) AS CapitalAmount,
    ABS(p.CommissionAmount) AS CommissionAmount,
    ABS(p.ExpenseAmount) AS ExpenseAmount,
    p.InstallmentAmountPaid,
    p.CapitalAmountPaid,
    p.ExpenseAmountPaid,
    p.CommissionAmountPaid,
    p.InterestAmountPaid,
    ABS(p.RemainingAmount) AS RemainingAmount,
    p.DownPaymentFlag,
    p.SourceSystem,
    p.SettlementKey,
    p.SettlementCaseRecordKey,
    p.PayTolerance,
    p.PayPercent,
    p.PayPercentAmount,
    p.PartyKey,
    p.AgreementKey,
    p.RecordKey,
    p.PortfolioName,
    p.PortfolioKey,
    p.ImplementationFlag,
    p.BusinessDate
FROM  `{env}-cdh`.primary.paymentschedule as p
WHERE p.BusinessDate = {run_date!r} and p.SourceSystem='profits'
"""
) 

paymentscheduleprofits.createOrReplaceTempView("paymentscheduleprofits")

paymentscheduleqca = spark.sql(f""" select 
    p.ExposureKey, 
    p.InstallmentDate, 
    COALESCE(NULLIF(TRIM(p.InstallmentTypeCode), ''), 'N/A') AS InstallmentTypeCode,
    p.SettlementKey,
    p.SettlementCaseRecordKey,
    p.RecordKey,
    p.InstallmentNumber,
    p.RequestSN,
    p.PaymentScheduleTypeCode,
    p.OverdueDays, 
    COALESCE(NULLIF(TRIM(p.InstallmentStatusCode), ''), 'N/A') AS InstallmentStatusCode,
    p.InstallmentAmount,
    p.InterestAmount,
    p.CapitalAmount,
    p.CommissionAmount,
    p.ExpenseAmount,
    p.InstallmentAmountPaid,
    p.CapitalAmountPaid,
    p.ExpenseAmountPaid,
    p.CommissionAmountPaid,
    p.InterestAmountPaid,
    p.RemainingAmount,
    p.DownPaymentFlag,
    p.SourceSystem,
    p.RecordKey,
    p.PayTolerance,
    p.PayPercent,
    p.PayPercentAmount,
    p.PartyKey,
    p.AgreementKey,
    p.PortfolioName,
    p.PortfolioKey,
    p.PaymentScheduleType,
    COALESCE(NULLIF(TRIM(p.InstallmentType), ''), 'N/A') AS InstallmentType,
    COALESCE(NULLIF(TRIM(p.InstallmentStatus), ''), 'N/A') AS InstallmentStatus,
    p.ImplementationFlag,
    p.BusinessDate
FROM  `{env}-cdh`.primary.paymentschedule as p 
WHERE p.BusinessDate = {run_date!r} and p.SourceSystem='qca'
"""
) 

paymentscheduleqca.createOrReplaceTempView("paymentscheduleqca")

In [0]:
#find duplicate rows based on the key values (ACCOUNT_NUMBER,REQUEST_DT,REQUEST_TYPE) and create a view
#only for profits!

paym = spark.sql(f""" SELECT
    ExposureKey, 
    SettlementKey,
    InstallmentNumber,
    InstallmentDate, 
    InstallmentTypeCode
FROM
    paymentscheduleprofits 
GROUP BY
    ExposureKey, 
    SettlementKey,
    InstallmentNumber,
    InstallmentDate, 
    InstallmentTypeCode
HAVING COUNT(*)>1
""")

paym.createOrReplaceTempView("paym")


In [0]:
# load settlementexposure to join it later.

settlementexposure_qca = spark.sql(f"""
    WITH filtered_settl AS (
        SELECT 
            CASE 
            WHEN prefix is null THEN settl.ExposureKey ELSE CONCAT(prefix,settl.ExposureKey) END as ExposureKey,
            SettlementKey,
            SourceSystem,
            ROW_NUMBER() OVER (PARTITION BY ExposureKey ORDER BY SubmissionDate DESC) AS rown
        FROM settlementexposure as settl
        LEFT JOIN portfolio p 
        ON p.portfolioname = settl.portfolioname
        WHERE 
            settl.SettlementStatus IN ('Running', 'Partially Fulfilled')
            AND settl.SettlementType != 'Νόμος 3869 – Προσωρινή Διαταγή'
            AND settl.SourceSystem = 'qca'
            AND settl.isactiveflag = 1
    )
    SELECT 
        ExposureKey, 
        SettlementKey,
        SourceSystem
    FROM filtered_settl
    WHERE rown = 1
""")

settlementexposure_qca.createOrReplaceTempView("settlementexposure_qca")

In [0]:
#group by these duplicate rows based on key and PortfolioId,rq_overdiew_days,REQUEST_STATUS and create another temp view

paymentscheduleprofits_groupedby1 = spark.sql(f"""
    SELECT
        ExposureKey, 
        InstallmentDate, 
        InstallmentTypeCode,
        InstallmentType,
        NULL AS InstallmentNumber,
        NULL AS RequestSN,
        PaymentScheduleTypeCode,
        PaymentScheduleType,
        OverdueDays, 
        InstallmentStatusCode,
        InstallmentStatus,
        SUM(ABS(InstallmentAmount)) AS InstallmentAmount,
        SUM(ABS(InterestAmount)) AS InterestAmount,
        SUM(ABS(CapitalAmount)) AS CapitalAmount,
        SUM(ABS(CommissionAmount)) AS CommissionAmount,
        SUM(ABS(ExpenseAmount)) AS ExpenseAmount,
        SUM(CapitalAmountPaid + ExpenseAmountPaid + CommissionAmountPaid + InterestAmountPaid) AS InstallmentAmountPaid,
        SUM(CapitalAmountPaid) AS CapitalAmountPaid,
        SUM(ExpenseAmountPaid) AS ExpenseAmountPaid,
        SUM(CommissionAmountPaid) AS CommissionAmountPaid,
        SUM(InterestAmountPaid) AS InterestAmountPaid,
        SUM(ABS(RemainingAmount)) AS RemainingAmount,
        NULL AS DownPaymentFlag,
        SourceSystem,
        SettlementKey,
        SettlementCaseRecordKey,
        RecordKey,
        PayTolerance,
        PayPercent,
        PayPercentAmount,
        PartyKey,
        AgreementKey,
        PortfolioName,
        PortfolioKey,
        ImplementationFlag,
        BusinessDate
    FROM
        paymentscheduleprofits AS p
    INNER JOIN (SELECT ExposureKey ekey, InstallmentDate idate, InstallmentTypeCode icode FROM paym) paym 
        ON p.ExposureKey = paym.ekey
        AND p.InstallmentDate = paym.idate
        AND p.InstallmentTypeCode = paym.icode
    GROUP BY
        ExposureKey, 
        InstallmentDate, 
        InstallmentTypeCode,
        PaymentScheduleTypeCode,
        OverdueDays, 
        InstallmentStatusCode,
        BusinessDate,
        SourceSystem,
        SettlementKey,
        SettlementCaseRecordKey,
        RecordKey,
        PayTolerance,
        PayPercent,
        PayPercentAmount,
        PartyKey,
        AgreementKey,
        PortfolioName,
        PortfolioKey,
        PaymentScheduleType,
        InstallmentType,
        InstallmentStatus,
        ImplementationFlag
""")


paymentscheduleprofits_groupedby1.createOrReplaceTempView("paymentscheduleprofits_groupedby1")


In [0]:
#find duplicate rows based on the above grouped by result

double = spark.sql(f""" SELECT
    ExposureKey, 
    SettlementKey,
    InstallmentNumber,
    InstallmentDate, 
    InstallmentTypeCode
FROM
    paymentscheduleprofits_groupedby1 
GROUP BY
    ExposureKey, 
    SettlementKey,
    InstallmentNumber,
    InstallmentDate, 
    InstallmentTypeCode
HAVING COUNT(*)>1
""")

double.createOrReplaceTempView("double")

In [0]:
#group by these duplicate rows based on same columns as the first time, but now remove OverdueDays and create another temp view

paymentscheduleprofits_groupedby2 = spark.sql(f""" 
    SELECT
        ExposureKey, 
        InstallmentDate, 
        InstallmentTypeCode,
        InstallmentType,
        NULL AS InstallmentNumber,
        NULL AS RequestSN,
        PaymentScheduleTypeCode,
        PaymentScheduleType,
        sum(OverdueDays) AS OverdueDays,
        InstallmentStatusCode,
        InstallmentStatus,
        SUM(ABS(InstallmentAmount)) AS InstallmentAmount,
        SUM(ABS(InterestAmount)) AS InterestAmount,
        SUM(ABS(CapitalAmount)) AS CapitalAmount,
        SUM(ABS(CommissionAmount)) AS CommissionAmount,
        SUM(ABS(ExpenseAmount)) AS ExpenseAmount,
        sum(CapitalAmountPaid + ExpenseAmountPaid + CommissionAmountPaid + InterestAmountPaid) AS InstallmentAmountPaid,
        sum(CapitalAmountPaid) AS CapitalAmountPaid,
        sum(ExpenseAmountPaid) AS ExpenseAmountPaid,
        sum(CommissionAmountPaid) AS CommissionAmountPaid,
        sum(InterestAmountPaid) AS InterestAmountPaid,
        sum(ABS(RemainingAmount)) AS RemainingAmount,
        NULL AS DownPaymentFlag,
        SourceSystem,
        SettlementKey,
        SettlementCaseRecordKey,
        RecordKey,
        PayTolerance,
        PayPercent,
        PayPercentAmount,
        PartyKey,
        AgreementKey,
        PortfolioName,
        PortfolioKey,
        ImplementationFlag,
        BusinessDate
    FROM
        paymentscheduleprofits AS p
    INNER JOIN (SELECT ExposureKey ekey, InstallmentDate idate, InstallmentTypeCode icode FROM double) double 
        ON p.ExposureKey = double.ekey
        AND p.InstallmentDate = double.idate
        AND p.InstallmentTypeCode = double.icode
    GROUP BY
        ExposureKey, 
        InstallmentDate, 
        InstallmentTypeCode,
        PaymentScheduleTypeCode,
        InstallmentStatus,
        InstallmentStatusCode,
        BusinessDate,
        SourceSystem,
        SettlementKey,
        SettlementCaseRecordKey,
        RecordKey,
        PayTolerance,
        PayPercent,
        PayPercentAmount,
        PartyKey,
        AgreementKey,
        PortfolioName,
        PortfolioKey,
        PaymentScheduleType,
        InstallmentType,
        ImplementationFlag
""")

paymentscheduleprofits_groupedby2.createOrReplaceTempView("paymentscheduleprofits_groupedby2")

In [0]:
#find duplicate rows based on the above grouped by result

triple = spark.sql(f""" SELECT
    ExposureKey, 
    SettlementKey,
    InstallmentNumber,
    InstallmentDate, 
    InstallmentTypeCode
FROM
    paymentscheduleprofits_groupedby2 
GROUP BY
    ExposureKey, 
    SettlementKey,
    InstallmentNumber,
    InstallmentDate, 
    InstallmentTypeCode
HAVING COUNT(*)>1
""")

triple.createOrReplaceTempView("triple")

In [0]:
#group by these duplicate rows based on same columns as the first time, but now remove InstallmentStatusCode and create another temp view

paymentscheduleprofits_groupedby3 = spark.sql(f""" 
    SELECT
        ExposureKey, 
        InstallmentDate, 
        InstallmentTypeCode,
        InstallmentType,
        NULL AS InstallmentNumber,
        NULL AS RequestSN,
        PaymentScheduleTypeCode,
        PaymentScheduleType,
        sum(OverdueDays) AS OverdueDays,
        '3' as InstallmentStatusCode,
        'Partial Paid' InstallmentStatus,
        SUM(ABS(InstallmentAmount)) AS InstallmentAmount,
        SUM(ABS(InterestAmount)) AS InterestAmount,
        SUM(ABS(CapitalAmount)) AS CapitalAmount,
        SUM(ABS(CommissionAmount)) AS CommissionAmount,
        SUM(ABS(ExpenseAmount)) AS ExpenseAmount,
        sum(CapitalAmountPaid + ExpenseAmountPaid + CommissionAmountPaid + InterestAmountPaid) AS InstallmentAmountPaid,
        sum(CapitalAmountPaid) AS CapitalAmountPaid,
        sum(ExpenseAmountPaid) AS ExpenseAmountPaid,
        sum(CommissionAmountPaid) AS CommissionAmountPaid,
        sum(InterestAmountPaid) AS InterestAmountPaid,
        sum(ABS(RemainingAmount)) AS RemainingAmount,
        NULL AS DownPaymentFlag,
        SourceSystem,
        SettlementKey,
        SettlementCaseRecordKey,
        RecordKey,
        PayTolerance,
        PayPercent,
        PayPercentAmount,
        PartyKey,
        AgreementKey,
        PortfolioName,
        PortfolioKey,
        ImplementationFlag,
        BusinessDate
    FROM
        paymentscheduleprofits AS p
    INNER JOIN (SELECT ExposureKey ekey, InstallmentDate idate, InstallmentTypeCode icode FROM triple) triple 
        ON p.ExposureKey = triple.ekey
        AND p.InstallmentDate = triple.idate
        AND p.InstallmentTypeCode = triple.icode
    GROUP BY
        ExposureKey, 
        InstallmentDate, 
        InstallmentTypeCode,
        PaymentScheduleTypeCode,
        BusinessDate,
        SourceSystem,
        SettlementKey,
        SettlementCaseRecordKey,
        RecordKey,
        PayTolerance,
        PayPercent,
        PayPercentAmount,
        PartyKey,
        AgreementKey,
        PortfolioName,
        PortfolioKey,
        PaymentScheduleType,
        InstallmentType,
        ImplementationFlag
""")

paymentscheduleprofits_groupedby3.createOrReplaceTempView("paymentscheduleprofits_groupedby3")

In [0]:
# Make a union of:
# 1. Values that do not have duplicates (from profits)
# 2. Duplicates that were resolved by grouping with mupltiple columns including REQUEST_STATUS (from profits) (rows with different type)
# 3. Duplicates that were resolved by grouping with REQUEST_STATUS (from profits) but without overdue days (rows with same type/status but different overdue days)
# 4. Duplicates that were resolved by grouping without REQUEST_STATUS (from profits) (rows with different status)

target_table_dpaymentscheduleprofits = spark.sql(f""" 
SELECT paymentscheduleprofits.*
FROM paymentscheduleprofits
LEFT JOIN paym
ON paymentscheduleprofits.ExposureKey = paym.ExposureKey
AND paymentscheduleprofits.InstallmentDate = paym.InstallmentDate
AND paymentscheduleprofits.InstallmentTypeCode = paym.InstallmentTypeCode
WHERE paym.ExposureKey IS NULL

UNION

SELECT paymentscheduleprofits_groupedby1.* 
FROM paymentscheduleprofits_groupedby1
LEFT JOIN double
ON paymentscheduleprofits_groupedby1.ExposureKey = double.ExposureKey
AND paymentscheduleprofits_groupedby1.InstallmentDate = double.InstallmentDate
AND paymentscheduleprofits_groupedby1.InstallmentTypeCode = double.InstallmentTypeCode
WHERE double.ExposureKey IS NULL

UNION

SELECT paymentscheduleprofits_groupedby2.* 
FROM paymentscheduleprofits_groupedby2
LEFT JOIN triple
ON paymentscheduleprofits_groupedby2.ExposureKey = triple.ExposureKey
AND paymentscheduleprofits_groupedby2.InstallmentDate = triple.InstallmentDate
AND paymentscheduleprofits_groupedby2.InstallmentTypeCode = triple.InstallmentTypeCode
WHERE triple.ExposureKey IS NULL

UNION

SELECT * FROM paymentscheduleprofits_groupedby3

""")

# Create or replace temp view for the final DataFrame
target_table_dpaymentscheduleprofits.createOrReplaceTempView("target_table_dpaymentscheduleprofits")

In [0]:
target_table_dpaymentscheduleprofits = spark.sql(f"""
          SELECT ttdp.* from
          target_table_dpaymentscheduleprofits ttdp
          left join settlementexposure_qca qca ON ttdp.ExposureKey = qca.ExposureKey 
          WHERE qca.ExposureKey IS NULL
          """)

target_table_dpaymentscheduleprofits.createOrReplaceTempView("target_table_dpaymentscheduleprofits")


In [0]:
 
# Calculate key for profit and create a dataframe for records received from profit source system

target_table_dpaymentscheduleprofitsid_primary_key_fields = [
    target_table_dpaymentscheduleprofits["ExposureKey"],
    target_table_dpaymentscheduleprofits["SettlementKey"],
    target_table_dpaymentscheduleprofits["InstallmentNumber"],
    target_table_dpaymentscheduleprofits["InstallmentDate"],
    target_table_dpaymentscheduleprofits["InstallmentTypeCode"]
]

target_table_dpaymentscheduleprofitsKey_primary_key_fields = [
    target_table_dpaymentscheduleprofits["ExposureKey"],
    target_table_dpaymentscheduleprofits["SettlementKey"],
    target_table_dpaymentscheduleprofits["InstallmentNumber"],
    target_table_dpaymentscheduleprofits["InstallmentDate"],
    target_table_dpaymentscheduleprofits["InstallmentTypeCode"]
]

target_table_dpaymentscheduleprofits = (
    target_table_dpaymentscheduleprofits.withColumn(
        "DPaymentScheduleId",
        sha2(
            concat_ws(";", *target_table_dpaymentscheduleprofitsid_primary_key_fields),
            256,
        ),
    ).withColumn(
        "DPaymentScheduleKey",
        concat_ws(";", *target_table_dpaymentscheduleprofitsKey_primary_key_fields)
    )
)

result_df_profits = target_table_dpaymentscheduleprofits.select(
    target_table_dpaymentscheduleprofits.DPaymentScheduleId.cast("String"),
    target_table_dpaymentscheduleprofits.PortfolioName.cast("String"),
    target_table_dpaymentscheduleprofits.PortfolioKey.cast("int"),
    lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
        "IngestionTimestamp"
    ),
    lit(None).alias("BatchId"),
    target_table_dpaymentscheduleprofits.SourceSystem.cast("string"),
    target_table_dpaymentscheduleprofits.DPaymentScheduleKey.cast("string"),
    target_table_dpaymentscheduleprofits.ExposureKey.cast("string"), 
    target_table_dpaymentscheduleprofits.PartyKey.cast("string"),
    target_table_dpaymentscheduleprofits.AgreementKey.cast("string"),
    target_table_dpaymentscheduleprofits.InstallmentDate.cast("timestamp"), 
    target_table_dpaymentscheduleprofits.SettlementKey.cast("string"), 
    target_table_dpaymentscheduleprofits.SettlementCaseRecordKey.cast("int"), 
    target_table_dpaymentscheduleprofits.PayTolerance.cast("int"), 
    target_table_dpaymentscheduleprofits.PayPercent.cast("decimal(12,6)"), 
    target_table_dpaymentscheduleprofits.PayPercentAmount.cast("decimal(18,2)"), 
    target_table_dpaymentscheduleprofits.RecordKey.cast("int"), 
    target_table_dpaymentscheduleprofits.InstallmentTypeCode.cast("string"),
    target_table_dpaymentscheduleprofits.InstallmentType.cast("string"),
    target_table_dpaymentscheduleprofits.InstallmentNumber.cast("int"),
    target_table_dpaymentscheduleprofits.RequestSN.cast("int"),
    target_table_dpaymentscheduleprofits.PaymentScheduleTypeCode.cast("string"),
    target_table_dpaymentscheduleprofits.PaymentScheduleType.cast("string"),
    target_table_dpaymentscheduleprofits.OverdueDays.cast("int"), 
    target_table_dpaymentscheduleprofits.InstallmentStatusCode.cast("string"),
    target_table_dpaymentscheduleprofits.InstallmentStatus.cast("string"),
    target_table_dpaymentscheduleprofits.InstallmentAmount.cast("Decimal(38,7)"),
    target_table_dpaymentscheduleprofits.InterestAmount.cast("Decimal(19,2)"),
    target_table_dpaymentscheduleprofits.CapitalAmount.cast("Decimal(19,2)"),
    target_table_dpaymentscheduleprofits.CommissionAmount.cast("Decimal(19,2)"),
    target_table_dpaymentscheduleprofits.ExpenseAmount.cast("Decimal(19,2)"),
    target_table_dpaymentscheduleprofits.InstallmentAmountPaid.cast("Decimal(19,2)"),
    target_table_dpaymentscheduleprofits.CapitalAmountPaid.cast("Decimal(19,2)"),
    target_table_dpaymentscheduleprofits.ExpenseAmountPaid.cast("Decimal(19,2)"),
    target_table_dpaymentscheduleprofits.CommissionAmountPaid.cast("Decimal(19,2)"),
    target_table_dpaymentscheduleprofits.InterestAmountPaid.cast("Decimal(19,2)"),
    target_table_dpaymentscheduleprofits.RemainingAmount.cast("Decimal(19,2)"),
    target_table_dpaymentscheduleprofits.DownPaymentFlag.cast("boolean"),
    target_table_dpaymentscheduleprofits.ImplementationFlag.cast("boolean"),
    target_table_dpaymentscheduleprofits.BusinessDate.alias("BusinessDate"),
)

In [0]:
# Spark SQL query
exposure_keys_for_qca = """
SELECT 
    sett.ExposureKey, 
    p.InstallmentDate, 
    COALESCE(NULLIF(TRIM(p.InstallmentTypeCode), ''), 'N/A') AS InstallmentTypeCode,
    p.SettlementKey,
    p.SettlementCaseRecordKey,
    p.RecordKey,
    p.InstallmentNumber,
    p.RequestSN,
    p.PaymentScheduleTypeCode,
    p.OverdueDays, 
    COALESCE(NULLIF(TRIM(p.InstallmentStatusCode), ''), 'N/A') AS InstallmentStatusCode,
    p.InstallmentAmount,
    p.InterestAmount,
    p.CapitalAmount,
    p.CommissionAmount,
    p.ExpenseAmount,
    p.InstallmentAmountPaid,
    p.CapitalAmountPaid,
    p.ExpenseAmountPaid,
    p.CommissionAmountPaid,
    p.InterestAmountPaid,
    p.RemainingAmount,
    p.DownPaymentFlag,
    p.SourceSystem,
    p.RecordKey,
    p.PayTolerance,
    p.PayPercent,
    p.PayPercentAmount,
    p.PartyKey,
    p.AgreementKey,
    p.PortfolioName,
    p.PortfolioKey,
    p.PaymentScheduleType,
    COALESCE(NULLIF(TRIM(p.InstallmentType), ''), 'N/A') AS InstallmentType,
    COALESCE(NULLIF(TRIM(p.InstallmentStatus), ''), 'N/A') AS InstallmentStatus,
    p.ImplementationFlag,
    p.BusinessDate
FROM paymentscheduleqca p
INNER JOIN settlementexposure_qca sett
    ON sett.SettlementKey = p.SettlementKey
    AND sett.SourceSystem = p.SourceSystem
"""

# Execute the query
paymentscheduleqca = spark.sql(exposure_keys_for_qca)

In [0]:
# Calculate key for qca and create a dataframe for records received from qca source system

target_table_dpaymentscheduleqcaid_primary_key_fields = [
    paymentscheduleqca["Exposurekey"],
    paymentscheduleqca["SettlementKey"],
    paymentscheduleqca["RecordKey"],
    paymentscheduleqca["SettlementCaseRecordKey"]
]

target_table_dpaymentscheduleqcaKey_primary_key_fields = [
    paymentscheduleqca["Exposurekey"],
    paymentscheduleqca["SettlementKey"],
    paymentscheduleqca["RecordKey"],
    paymentscheduleqca["SettlementCaseRecordKey"]
]

target_table_dpaymentscheduleqca = (
    paymentscheduleqca.withColumn(
        "dpaymentscheduleqcaId",
        sha2(
            concat_ws(";", *target_table_dpaymentscheduleqcaid_primary_key_fields),
            256,
        ),
    ).withColumn(
        "dpaymentscheduleqcaKey",
        concat_ws(";", *target_table_dpaymentscheduleqcaKey_primary_key_fields)
    )
)

result_df_qca = target_table_dpaymentscheduleqca.select(
    target_table_dpaymentscheduleqca.dpaymentscheduleqcaId.cast("String"),
    target_table_dpaymentscheduleqca.PortfolioName.cast("String"),
    target_table_dpaymentscheduleqca.PortfolioKey.cast("int"),
    lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
        "IngestionTimestamp"
    ),
    lit(None).alias("BatchId"),
    target_table_dpaymentscheduleqca.SourceSystem.cast("string"),
    target_table_dpaymentscheduleqca.dpaymentscheduleqcaKey.cast("string"),
    target_table_dpaymentscheduleqca.ExposureKey.cast("string"), 
    target_table_dpaymentscheduleqca.PartyKey.cast("string").alias("PartyKey"),
    target_table_dpaymentscheduleqca.AgreementKey.cast("string").alias("AgreementKey"),
    target_table_dpaymentscheduleqca.InstallmentDate.cast("timestamp"), 
    target_table_dpaymentscheduleqca.SettlementKey.cast("string"), 
    target_table_dpaymentscheduleqca.SettlementCaseRecordKey.cast("int"), 
    target_table_dpaymentscheduleqca.PayTolerance.cast("int").alias("PayTolerance"), 
    target_table_dpaymentscheduleqca.PayPercent.cast("decimal(12,6)").alias("PayPercent"), 
    target_table_dpaymentscheduleqca.PayPercentAmount.cast("decimal(18,2)").alias("PayPercentAmount"), 
    target_table_dpaymentscheduleqca.RecordKey.cast("int"), 
    target_table_dpaymentscheduleqca.InstallmentTypeCode.cast("string"),
    target_table_dpaymentscheduleqca.InstallmentType.cast("string"),
    target_table_dpaymentscheduleqca.InstallmentNumber.cast("int"),
    target_table_dpaymentscheduleqca.RequestSN.cast("int"),
    target_table_dpaymentscheduleqca.PaymentScheduleTypeCode.cast("string"),
    target_table_dpaymentscheduleqca.PaymentScheduleType.cast("string"),
    target_table_dpaymentscheduleqca.OverdueDays.cast("int"), 
    target_table_dpaymentscheduleqca.InstallmentStatusCode.cast("string"),
    target_table_dpaymentscheduleqca.InstallmentStatus.cast("string"),
    target_table_dpaymentscheduleqca.InstallmentAmount.cast("Decimal(38,7)"),
    target_table_dpaymentscheduleqca.InterestAmount.cast("Decimal(19,2)"),
    target_table_dpaymentscheduleqca.CapitalAmount.cast("Decimal(19,2)"),
    target_table_dpaymentscheduleqca.CommissionAmount.cast("Decimal(19,2)"),
    target_table_dpaymentscheduleqca.ExpenseAmount.cast("Decimal(19,2)"),
    target_table_dpaymentscheduleqca.InstallmentAmountPaid.cast("Decimal(19,2)"),
    target_table_dpaymentscheduleqca.CapitalAmountPaid.cast("Decimal(19,2)"),
    target_table_dpaymentscheduleqca.ExpenseAmountPaid.cast("Decimal(19,2)"),
    target_table_dpaymentscheduleqca.CommissionAmountPaid.cast("Decimal(19,2)"),
    target_table_dpaymentscheduleqca.InterestAmountPaid.cast("Decimal(19,2)"),
    target_table_dpaymentscheduleqca.RemainingAmount.cast("Decimal(19,2)"),
    target_table_dpaymentscheduleqca.DownPaymentFlag.cast("boolean"),
    target_table_dpaymentscheduleqca.ImplementationFlag.cast("boolean"),
    target_table_dpaymentscheduleqca.BusinessDate.alias("BusinessDate"),
)

In [0]:
result_df = result_df_profits.union(result_df_qca)


In [0]:
if unity == "True":
    result_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(derivative_table_path) # in order to write in delta_lake
else: 
    result_df.write \
        .format("delta") \
        .mode("append") \
        .save(derivative_table_path) # in order to write in delta_lake 